# 目标检测

## 介绍

图像识别为整个图像块分配单个标签，而语义分割标记每个像素但无法区分单个对象。许多地理空间应用——如车辆计数、船只定位或太阳能电池板盘点——需要分别识别和定位每个对象。

目标检测通过生成一组边界框来解决这个问题，每个边界框带有类别标签和置信度分数，用于定位图像中的单个对象。在地理空间环境中，这些边界框可以进行地理参考并转换为GIS工作流的矢量特征。

本教程涵盖地理空间图像目标检测的基础，包括关键概念、主要架构系列，以及使用`geoai`包在[NWPU-VHR-10](https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip)数据集上训练和评估Faster R-CNN v2检测器的实践工作流。

## 学习目标

在本教程结束时，您将能够：

- 解释目标检测与图像分类和语义分割的区别
- 描述边界框、置信度分数、非极大值抑制和锚框
- 比较两阶段、单阶段、基于Transformer和零样本检测架构
- 准备COCO注释格式的检测数据集
- 使用`geoai`包训练多类别目标检测模型
- 使用COCO风格的平均精度（mAP）评估检测性能
- 在新图像上运行推理并可视化检测结果
- 将训练好的模型发布到Hugging Face Hub并从托管模型运行推理

## 理解目标检测

### 分类与检测

图像分类为每个图像分配一个标签（例如"住宅区"），但不说明特定对象在哪里或有多少个。目标检测输出可变数量的边界框，每个边界框带有类别标签和置信度分数，支持对单个对象进行计数和定位。

语义分割提供像素级标签但不分离单个对象。目标检测提供对象级定位但不描绘精确边界。实例分割结合了这两种能力。

### 关键概念

**边界框**是检测模型的基本输出。每个框由指定检测对象周围矩形的四个坐标定义，通常采用角点格式（x_min, y_min, x_max, y_max）或中心格式（center_x, center_y, width, height）。在地理空间应用中，像素坐标通过图像的仿射变换转换为地理坐标。

**置信度分数**范围从0到1，表示模型对每个检测的确定性。高置信度阈值减少误报但可能遗漏对象，而低阈值捕获更多对象但引入更多误检。

**交并比（IoU）**通过交集面积除以并集面积来衡量预测框和真实框之间的重叠。它在训练和评估期间用于确定检测是否正确。

**非极大值抑制（NMS）**通过保留最高置信度框并抑制超过IoU阈值的其他框来消除冗余重叠检测。这对于密集场景中的干净输出至关重要。

**锚框**是放置在图像上的各种尺度和宽高比的预定义边界框模板。模型预测相对于这些锚点的偏移量，与预测绝对坐标相比简化了学习过程。

## 检测架构

### 两阶段检测器

两阶段检测器将检测分为候选区域生成和分类。**Faster R-CNN**首先使用区域候选网络（RPN）识别候选区域，然后通过单独的头部分类和细化每个候选区域。

两阶段检测器实现高精度并很好地处理多尺度，使其适用于地理空间图像。然而，其顺序设计使其比单阶段替代方案慢。本教程使用带有ResNet-50 FPN骨干的Faster R-CNN v2。

### 单阶段检测器

单阶段检测器在一次传递中直接从特征图预测边界框和类别标签，使其比两阶段方法快得多。

- **YOLO**（You Only Look Once）将检测构建为网格上的密集预测，现代变体使用多尺度特征金字塔。
- **SSD**（Single Shot MultiBox Detector）从不同分辨率的多个特征图进行预测以处理不同大小的对象。
- **RetinaNet**使用焦点损失通过专注于困难示例的训练来解决类别不平衡。在`geoai`中支持为`retinanet_resnet50_fpn_v2`。
- **FCOS**是一种无锚框检测器，直接在每个空间位置预测框。在`geoai`中支持为`fcos_resnet50_fpn`。

单阶段检测器适用于实时应用和处理大量图像。

### 基于Transformer的检测器

[DETR](https://huggingface.co/docs/transformers/model_doc/detr)（DEtection TRansformer）使用Transformer编码器-解码器架构将检测构建为集合预测问题。它通过在训练期间使用学习到的对象查询和匈牙利匹配来消除锚框和NMS。

DETR擅长全局上下文推理但收敛缓慢，并且在处理非常小的对象时存在困难。像Deformable DETR、DINO和RT-DETR这样的变体解决了这些限制。

### 零样本检测

零样本检测器识别由文本提示描述的对象，无需特定任务的训练数据。[OWL-ViT](https://huggingface.co/docs/transformers/en/model_doc/owlvit)结合了视觉Transformer和文本编码器，支持通过"太阳能电池板"或"游泳池"等自然语言查询进行检测。

[Grounding DINO](https://huggingface.co/docs/transformers/en/model_doc/grounding-dino)通过结合基于DINO的检测和接地语言理解来扩展这一点，实现强大的零样本性能。当没有标注训练数据时，这些模型对于快速原型开发非常宝贵。

### 选择架构

选择取决于您的应用需求：

- **Faster R-CNN**是优先考虑准确性时的强大默认选择。它是`geoai`包中的默认选项。
- **YOLO**在处理速度很重要时是首选，例如扫描大型档案或近实时监控。
- **DETR及其变体**适用于需要全局上下文推理的场景，避免锚框调优。
- **零样本检测器**（OWL-ViT、Grounding DINO）适用于探索性分析或没有标注数据时。

`geoai`包通过`model_name`参数支持多种检测架构：

| Model Name                          | Type         | Notes                                  |
| ----------------------------------- | ------------ | -------------------------------------- |
| `fasterrcnn_resnet50_fpn_v2`        | 两阶段       | 默认，良好的准确性/速度权衡 |
| `fasterrcnn_mobilenet_v3_large_fpn` | Two-stage    | Fastest two-stage option               |
| `retinanet_resnet50_fpn_v2`         | Single-stage | Fast, handles class imbalance well     |
| `fcos_resnet50_fpn`                 | Single-stage | Anchor-free                            |
| `maskrcnn_resnet50_fpn`             | 两阶段       | 还生成实例掩码（最慢）   |

将检测器的优势与目标对象相对于图像分辨率的尺度相匹配。只跨越几个像素的对象需要具有强大小对象能力的架构（例如特征金字塔网络），而大多数架构都能充分检测大对象。

## 准备检测数据集

### 注释格式

**COCO格式**将所有注释存储在单个JSON文件中，包含图像元数据、类别定义和边界框（以绝对像素坐标表示为(x, y, width, height)）。这是`geoai`包使用的格式。

**YOLO格式**每个图像使用一个文本文件，包含归一化坐标（`class_id center_x center_y width height`），使其轻量且易于编写脚本。

### NWPU-VHR-10数据集

[NWPU-VHR-10](https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip)数据集是超高分辨率遥感图像目标检测的基准数据集，包含650个标注图像和150个仅背景图像，涵盖10个类别：飞机、船只、储罐、棒球场、网球场、篮球场、田径场、港口、桥梁和车辆。

`geoai`包提供`prepare_nwpu_vhr10`来自动将标注图像分割为训练集和验证集，并生成COCO格式的注释。

## 评估检测结果

### 平均精度（mAP）

如果检测与真实框的IoU超过阈值且预测正确的类别，则为**真阳性**；否则为**假阳性**。未匹配的真实框是**假阴性**。

单个类别的**平均精度（AP）**是精确率-召回率曲线下的面积。**平均精度均值（mAP）**是所有类别的AP的平均值。

### 精确率-召回率曲线

精确率-召回率曲线绘制置信度阈值变化时的精确率与召回率。强大的检测器在高召回率时保持高精度。每类曲线揭示模型的难点，指导数据收集和增强决策。

### IoU阈值

- **mAP@0.5**要求50%重叠（传统的PASCAL VOC指标，在地理空间应用中很常见）。
- **mAP@0.75**要求75%重叠，奖励精确的定位。
- **mAP@0.5:0.95**在0.5到0.95的阈值范围内以0.05为步长平均（主要的COCO指标）。

合适的指标取决于您的用例：mAP@0.5可能足以满足对象计数，而更严格的阈值适用于需要精确定位的应用。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install "geoai-py[extra]"

## 导入库

`geoai`包为完整的目标检测管道提供函数：

- `download_file`：下载并提取数据集归档
- `NWPU_VHR10_CLASSES`：10个对象类别名称加上背景的列表
- `prepare_nwpu_vhr10`：将正图像分割为训练/验证集并生成COCO注释
- `visualize_coco_annotations`：显示带有标注边界框的示例图像
- `train_multiclass_detector`：训练Faster R-CNN或其他检测模型
- `plot_detection_training_history`：绘制训练时期的损失曲线
- `evaluate_multiclass_detector`：计算COCO风格的mAP指标
- `multiclass_detection`：在单个图像上运行推理
- `visualize_multiclass_detections`：在图像上显示检测结果
- `batch_multiclass_detection`：在多个图像上运行推理
- `push_detector_to_hub`：将训练好的模型上传到Hugging Face Hub
- `predict_detector_from_hub`：使用Hub托管的模型运行推理

In [ ]:
import os
import json

import geoai

## 下载NWPU-VHR-10数据集

从Source Cooperative下载并提取数据集。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip"
data_dir = geoai.download_file(url)

In [ ]:
print(f"Dataset directory: {data_dir}")
print(f"Contents: {os.listdir(data_dir)}")

## 探索数据集

数据集包含10个对象类别以及索引0处的背景类别。

In [ ]:
print(f"\nNWPU-VHR-10 Classes:")
for i, name in enumerate(geoai.NWPU_VHR10_CLASSES):
    print(f"  {i}: {name}")

```text
NWPU-VHR-10类别：
  0: background
  1: airplane
  2: ship
  3: storage_tank
  4: baseball_diamond
  5: tennis_court
  6: basketball_court
  7: ground_track_field
  8: harbor
  9: bridge
  10: vehicle
```

## 准备数据集

将正图像分割为训练集和验证集，并生成COCO格式的注释。

In [ ]:
splits = geoai.prepare_nwpu_vhr10(data_dir, val_split=0.2, seed=42)

In [ ]:
print(f"Images directory: {splits['images_dir']}")
print(f"Number of classes: {splits['num_classes']}")
print(f"Class names: {splits['class_names']}")
print(f"Training images: {len(splits['train_image_ids'])}")
print(f"Validation images: {len(splits['val_image_ids'])}")

```text
图像目录：./NWPU-VHR-10/positive_image_set
类别数量：11
类别名称：['background', 'airplane', 'ship', 'storage_tank', 'baseball_diamond', 'tennis_court', 'basketball_court', 'ground_track_field', 'harbor', 'bridge', 'vehicle']
训练图像：509
验证图像：128
```

## 可视化示例注释

在训练前可视化带有真实边界框的示例图像以验证注释。

In [ ]:
geoai.visualize_coco_annotations(
    annotations_path=splits["annotations_path"],
    images_dir=splits["images_dir"],
    num_samples=6,
    random=True,
    seed=1,
    cols=3,
    figsize=(12, 6),
)

## 训练多类别检测模型

`train_multiclass_detector`函数处理完整的训练管道。关键参数：

- `model_name`：检测架构（默认`fasterrcnn_resnet50_fpn_v2`）。其他选项：`fasterrcnn_mobilenet_v3_large_fpn`、`retinanet_resnet50_fpn_v2`、`fcos_resnet50_fpn`、`maskrcnn_resnet50_fpn`。
- `class_names`：类别名称列表，索引0处包含背景。
- `pretrained=True`：使用ImageNet权重初始化骨干网络进行迁移学习。
- `num_epochs`：训练集的遍历次数。
- `batch_size`和`learning_rate`：标准优化超参数。
- `val_split`: fraction of training split held out for internal validation.

In [ ]:
output_dir = "nwpu_output"

model_path = geoai.train_multiclass_detector(
    images_dir=splits["images_dir"],
    annotations_path=splits["train_annotations"],
    output_dir=output_dir,
    model_name="fasterrcnn_resnet50_fpn_v2",
    class_names=splits["class_names"],
    num_channels=3,
    batch_size=4,
    num_epochs=10,
    learning_rate=0.005,
    val_split=0.1,
    seed=42,
    pretrained=True,
    verbose=True,
)

## 绘制训练指标

绘制训练损失、验证IoU和学习率调度随epoch的变化。

In [ ]:
geoai.plot_detection_training_history(
    history_path=os.path.join(output_dir, "training_history.pth"),
)

## 使用COCO指标评估

在验证集上运行训练好的模型，并计算所有10个对象类别的COCO风格mAP指标。

In [ ]:
metrics = geoai.evaluate_multiclass_detector(
    model_path=model_path,
    images_dir=splits["images_dir"],
    annotations_path=splits["val_annotations"],
    num_classes=splits["num_classes"],
    class_names=splits["class_names"][1:],  # Exclude background
    batch_size=4,
)

```text
评估结果：
  mAP@0.5:        0.7312
  mAP@0.75:       0.4936
  mAP@[0.5:0.95]: 0.4428

  每类AP@0.5：
    AP@0.5/airplane: 0.7106
    AP@0.5/baseball_diamond: 0.7885
    AP@0.5/basketball_court: 0.8957
    AP@0.5/bridge: 0.9052
    AP@0.5/ground_track_field: 0.7081
    AP@0.5/harbor: 0.5322
    AP@0.5/ship: 0.6349
    AP@0.5/storage_tank: 0.5624
    AP@0.5/tennis_court: 0.8967
    AP@0.5/vehicle: 0.6781
```

像篮球场和网球场这样大而独特的对象获得更高的AP，而像车辆、船只和储罐这样较小或更模糊的对象得分较低。

## 在示例图像上运行推理

使用`multiclass_detection`在验证图像上运行推理，它处理切片、NMS和结果组装。

In [ ]:
# Load validation data to pick a test image
with open(splits["val_annotations"], "r") as f:
    val_data = json.load(f)

test_img_info = val_data["images"][0]
test_img_path = os.path.join(splits["images_dir"], test_img_info["file_name"])
print(f"Test image: {test_img_path}")

In [ ]:
output_raster = "nwpu_detection_output.tif"

result_path, inference_time, detections = geoai.multiclass_detection(
    input_path=test_img_path,
    output_path=output_raster,
    model_path=model_path,
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
    window_size=512,
    overlap=256,
    confidence_threshold=0.5,
    batch_size=4,
    num_channels=3,
)

print(f"\nInference time: {inference_time:.2f}s")
print(f"Total detections: {len(detections)}")

```text
NMS前收集了27个检测
NMS后：7个检测
多类别检测在0.14秒内完成
最终检测：7
  harbor: 6 detections
  bridge: 1 detections
多类别检测保存到nwpu_detection_output.tif

推理时间：0.14秒
检测总数：7
```

## 可视化检测

在图像上叠加检测到的边界框，按类别着色并标注置信度分数。

In [ ]:
geoai.visualize_multiclass_detections(
    image_path=test_img_path,
    detections=detections,
    class_names=splits["class_names"],
    confidence_threshold=0.5,
    figsize=(12, 10),
)

## 多图像批量推理

一次性处理多个图像并在网格中可视化结果。

In [ ]:
val_image_paths = [
    os.path.join(splits["images_dir"], img["file_name"])
    for img in val_data["images"][:4]
]

results = geoai.batch_multiclass_detection(
    image_paths=val_image_paths,
    output_dir="nwpu_batch_output",
    model_path=model_path,
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
    confidence_threshold=0.5,
    num_channels=3,
    figsize=(16, 12),
)

预测并不完美——您可能会注意到假阳性（例如在水面上预测的棒球场）和假阴性（例如遗漏的船只）。

## 发布和重用模型

### 推送到Hugging Face Hub

将训练好的模型上传到Hugging Face Hub进行共享。您需要免费账户和写入访问令牌。

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
url = geoai.push_detector_to_hub(
    model_path=model_path,
    repo_id="your-username/nwpu-vhr10-fasterrcnn",
    model_name="fasterrcnn_resnet50_fpn_v2",
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
)

### 从Hub运行推理

从Hub下载模型并在没有本地检查点的情况下运行推理。

In [ ]:
sample_img_path = os.path.join(splits["images_dir"], "608.jpg")

result_path, inference_time, detections = geoai.predict_detector_from_hub(
    input_path=sample_img_path,
    output_path="hub_detection.tif",
    repo_id="giswqs/nwpu-vhr10-fasterrcnn",
    confidence_threshold=0.5,
)

print(f"Inference time: {inference_time:.2f}s")
print(f"Total detections: {len(detections)}")

# Clean up
if os.path.exists("hub_detection.tif"):
    os.remove("hub_detection.tif")

```text
NMS前收集了34个检测
NMS后：8个检测
多类别检测在0.13秒内完成
最终检测：8
  baseball_diamond: 1 detections
  tennis_court: 4 detections
  basketball_court: 3 detections
多类别检测保存到hub_detection.tif
推理时间：0.13秒
检测总数：8
```

In [ ]:
geoai.visualize_multiclass_detections(
    image_path=sample_img_path,
    detections=detections,
    class_names=geoai.NWPU_VHR10_CLASSES,
    confidence_threshold=0.5,
    figsize=(12, 10),
)

## 关键要点

1. 目标检测使用边界框定位和分类单个对象，填补了图像分类和语义分割之间的空白。

2. 边界框、IoU、NMS和锚框是检测模型如何生成、细化和过滤预测的基础概念。

3. 两阶段检测器（Faster R-CNN）优先考虑准确性，单阶段检测器（YOLO、RetinaNet、FCOS）优先考虑速度，基于Transformer的检测器（DETR）简化管道，零样本检测器（OWL-ViT）消除对训练数据的需求。

4. NWPU-VHR-10数据集为遥感多类别检测提供了10类基准，带有COCO格式注释。

5. `geoai`包简化了从数据集准备到训练、评估和推理的完整检测工作流。

6. 多个IoU阈值的COCO风格mAP指标提供全面评估，每类AP揭示优缺点。

7. 根据假阳性或假阴性哪个成本更高，置信度阈值调整平衡精确率和召回率。

8. Publishing models to Hugging Face Hub enables sharing and reuse without local training.